# An AI Who Makes Recipes

### I use two AIs here.  First, I use llama3.2:1b to find existing recipes on the internet.  Then I use gemma4:e2b to create a new recipe inspired by the recipes that were found.

## The first AI finds recipes on the internet.

### Import the necessary modules.

In [ ]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

### Set up Ollama.

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

### Get links from The Curry Guy.

In [ ]:
links = fetch_website_links("https://greatcurryrecipes.net")
links

### Set up a system prompt for finding existing recipes.

In [ ]:
existing_system_prompt = """
You are a strict, precise Recipe Extraction AI. Your primary goal is to extract the main culinary recipes 
from the text provided, while completely ignoring all surrounding webpage clutter.

STRICT INSTRUCTIONS:
1. Ignore all sidebar content, footer links, header navigation, trending recipe widgets, and 
"You Might Also Like" suggestions.
2. Focus ONLY on the primary recipe block data (usually contains ingredients, steps, and a matching title).
3. MATCH VALIDATION: You must verify that the recipe you extract explicitly matches the user's requested 
dish. 
4. CRITICAL REJECTION RULE: If the main recipe on the page is NOT the dish the user asked for (e.g., the 
user asked for Curry but the text is mainly about Chicken Noodle Soup), you MUST refuse to extract it. 
Return an empty JSON object {} and absolutely nothing else.
5. Do not apologize, explain yourself, or include any conversational prose. Output raw data only.

You should respond in JSON as in this example:
{
    "links": [
        {"dish": "green curry", "url": "https://full.url/goes/here/green"},
        {"dish": "Japanese curry", "url": "https://another.full.url/japanese"}
    ]
}

Please select 5 links at most.
"""

### Define a function for retrieving recipe links from greatcurryrecipes.net.

In [ ]:
def get_recipes(dish):
    user_prompt = """Here is a list of links from greatcurryrecipes.net.  
    Here is the dish that I would like recipes for: {dish}
    Please decide which of these links are relevant to the dish that I want.

    INSTRUCTIONS:
1. Analyze the links. 
2. Identify the three MOST RELEVANT links featured on this page.
3. If the most relevant recipes are NOT recipes for "{dish}" (even if "{dish}" is mentioned in the sidebar 
or comments), STOP and return {{}}.
"""
    recipes = fetch_website_links("https://greatcurryrecipes.net")
    user_prompt += "\n".join(links)
    return user_prompt

### Define a function that has the AI decide which links are relevant.

In [ ]:
def select_relevant_links(dish):
    response = ollama.chat.completions.create(
        model='llama3.2:1b',
        messages=[
            {"role": "system", "content": existing_system_prompt},
            {"role": "user", "content": get_recipes(dish)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

### Let's make sure everything works so far.

In [ ]:
select_relevant_links('curry')

## The second AI uses the recipes that have been found to make a new recipe.

### Set up new prompts for the AI chef.

In [ ]:
def fetch_page_and_all_relevant_links(url, dish):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(dish)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    
    # Use .get('links', []) so it defaults to an empty list instead of crashing
    links_list = relevant_links.get('links', []) if isinstance(relevant_links, dict) else []
    
    for link in links_list:
        # 1. Grab the URL string safely
        link_url = link.get('url', '')
        
        # 2. Make sure it actually looks like a web address before scraping
        if isinstance(link_url, str) and link_url.startswith(('http://', 'https://')):
            result += fetch_website_contents(link_url)
        else:
            result += f"\n* (Skipped an invalid link or a 'not found' message: {link_url})\n"
            
    return result

### Set up the system prompt.

In [ ]:
chef_system_prompt = """
You are a chef trained in Japanese, Chinese, French, and Scandinavian cuisine.  Your job is to take 
recipes from multiple webpages and combine ideas from all of them to come up with a new recipe.
The new recipe should consist of a bullet-pointed list of ingredients and how much of each to use (e.g. teaspoons,
tablespoons, cups), followed by step by step instructions for making this new dish.  The new recipe
should use a maximum of 10 ingredients.  You should also give the dish a name that reflects the main 
ingredients.
Respond in markdown without code blocks.
"""

### Set up the user prompt for the AI chef.

In [ ]:
url = "https://greatcurryrecipes.net"

def chef_user_prompt(url, dish):
    user_prompt = f"""
Please make a recipe for the following kind of dish: {dish}
Here are recipes that are similar to what I'm interested in.  Please use your knowledge of different
cuisines and the provided recipes to create a new recipe of the dish that I asked for.
Respond in markdown without code blocks.
Please include one sentence that describes how your new recipe incorporates ideas from the provided 
links.
"""
    user_prompt += fetch_page_and_all_relevant_links(url, dish)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

### Let's make sure this works.

In [ ]:
url = "https://greatcurryrecipes.net"
dish = 'curry'

chef_user_prompt(url, dish)

### Now we can have the AI create a new recipe.

In [ ]:
def create_recipe(url, dish):
    response = ollama.chat.completions.create(
        model="gemma4:e2b",
        messages=[
            {"role": "system", "content": chef_system_prompt},
            {"role": "user", "content": chef_user_prompt(url, dish)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

### Let's see what this chef is cooking!

In [ ]:
create_recipe('https://greatcurryrecipes.net', 'seafood')